# EVE 310 - Lecture notebook: Multivariate linear regression

**Module 2 | Lecture follow-along**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/lectures/notebooks/multivariate-linear-regression.ipynb)

This is the code shown in lecture for multivariate linear regression with the normal equations. Run each cell as it comes up, so the output on your screen matches the one in lecture.

## Learning objectives

By the end of this notebook you will be able to:

1. Build the matrix `X` from several features with `np.c_`, and add the column of ones for the intercept
2. Compute the coefficients with the normal equations, β = (XᵀX)⁻¹Xᵀy, using `np.dot`, `.T`, and `np.linalg.inv`
3. Predict for a new observation, and check the model with a predicted-vs-observed plot, a residual plot, and R² computed from SSE and SST

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run one cell at a time, top to bottom, as it comes up in lecture: click the cell and press **Shift + Enter**.
3. Keep to the order, and run the cell that adds the column of ones only once. The note there says what to do if you ran it twice.
4. There is nothing to fill in and nothing to submit.

## Water quality example

Five samples from a river. For each one you have the water temperature, the dissolved oxygen (DO), the flow rate, and the concentration measured downstream. The downstream concentrations are the ones from the univariate example, where the only input was the upstream concentration.

| Temperature (°C) | DO (mg/l) | Flow rate (MG/day) | Downstream concentration (mg/l) |
| ---- | ---- | ---- | ---- |
| 21 | 8 | 100 | 55 |
| 22 | 7 | 120 | 67 |
| 25 | 6.5 | 110 | 53 |
| 20 | 8 | 115 | 50 |
| 20 | 8.2 | 105 | 51 |

Find a model that estimates the downstream concentration from the three features:

h(x) = β0 + β1 × Temp + β2 × DO + β3 × Flow

In [ ]:
# Import libraries
import numpy as np #Data manipulation
import matplotlib.pyplot as plt # Visualization

## Create X and y

`y` is reshaped to a column vector, shape `(5, 1)`. `Temp`, `DO`, and `Flow` are 1-D arrays, shape `(5,)`.

In [ ]:
# create X, y
y = np.array([55,67,53,50,51]).reshape(-1,1)
Temp = np.array([21,22,25,20,20])
DO = np.array([8,7,6.5,8,8.2])
Flow = np.array([100,120,110,115,105])

`np.c_[...]` puts the 1-D arrays side by side as columns. `X` has one row per sample and one column per feature, so its shape is `(5, 3)`.

In [ ]:
# make a matrix by concatenating the arrays
X = np.c_[Temp,DO,Flow]
print(X)

The model has an intercept β0, and β0 needs a column to multiply. That column is all ones, and it goes first. After this cell `X` has shape `(5, 4)`.

**Run this cell once.** Each run adds one more column of ones. If you ran it twice, run the cell above (`X = np.c_[Temp,DO,Flow]`) again, then this cell once.

In [ ]:
# add a column of ones
X = np.c_[np.ones(X.shape[0]), X]
print(X)

## Normal equations

The coefficients that minimize the squared error are β = (XᵀX)⁻¹Xᵀy. The cell builds that product one piece at a time.

`beta` is a column vector of shape `(4, 1)`, in the order of the columns of `X`: the intercept β0 first, then the coefficients for `Temp`, `DO`, and `Flow`.

In [ ]:
# get betas
# np.dot  dot product between two arrays
# np.linalg.inv function to get the inverse of a matrix
# x.T - get transpose of x

xtx = np.dot(X.T, X)                 # x-transpose times x
inv_xtx = np.linalg.inv(xtx)         # inverse of x-transpose times x
xty = np.dot(X.T, y)                 # x-transpose times y
beta = np.dot(inv_xtx, xty)          # coefficients
print(beta)

## Predict a new value

A new sample has temperature 19, DO 8.5, and flow rate 107. It needs the same leading 1 as every row of `X`, so the intercept is counted. `.reshape(1, -1)` makes it a 1 × 4 row, and a 1 × 4 row times the 4 × 1 `beta` gives a 1 × 1 result: the predicted downstream concentration.

In [ ]:
# predict a new value
# e.g., x = [19,8.5,107], y = ?
######################

x_new = np.array([1,19,8.5,107]).reshape(1,-1)
y_predict = np.dot(x_new,beta)
print(y_predict)

## Plot the fit

`np.dot(X, beta)` predicts all five samples at once: 5 × 4 times 4 × 1 gives a 5 × 1 column. In the plot, each point is one sample, and the dashed gray line is where predicted = observed.

In [ ]:
# get predictions
ym = np.dot(X,beta)

# plot predicted vs. observedf = plt.figure()
plt.plot(y, ym, 'ob')  # plot observations
plt.plot(y, y, color='gray', linewidth=1,linestyle='dashed') # plot the perfect match line
plt.title('predicted vs observed: linear')
plt.xlabel('observed values')
plt.ylabel('predicted values')
plt.grid(True)

A residual is observed minus predicted. For a good model the residuals scatter around 0 with no pattern.

In [ ]:
# plot residuals vs. predicted values
# linear model
e_linear = y - ym

f = plt.figure()
plt.plot(ym,e_linear, 'or')  # plot observations
plt.title('residuals')
plt.xlabel('predicted values')
plt.ylabel('residuals')
plt.grid(True)

## Metrics to assess the model

SSE is the squared error the model leaves unexplained, and SST is the total variation of `y` around its mean. R² = 1 − SSE/SST, and the closer it is to 1, the better the fit.

This model has 4 coefficients and only 5 samples to fit them to, so a high R² here says little about how the model would do on new samples.

In [ ]:
# predict the values using the model we fitted previously
# we will need this to calculate the different metrics

# calculate the mean, we will need this to calculate the different metrics
avg_y = np.mean(y)

# SSE - unexplained errors
squared_errors = np.power(y - ym,2)
SSE = np.sum(squared_errors)
print('SSE: {:.2f}'.format(SSE))

# SST - total errors
squared_errors = (y - avg_y) ** 2
SST = np.sum(squared_errors)
print('SST: {:.2f}'.format(SST))

# R^2
R2 = 1 - SSE/SST
print('R^2: {:.2f}'.format(R2))

## Commands used

- `np.c_[a, b, c]`: put 1-D arrays side by side as the columns of a matrix
- `np.ones(n)`: an array of `n` ones
- `X.T`: the transpose of `X`
- `np.dot(A, B)`: the matrix product of `A` and `B`
- `np.linalg.inv(A)`: the inverse of a square matrix `A`
- `np.mean(y)`, `np.sum(...)`, `np.power(a, 2)`: the pieces of SSE, SST, and R²